## 1. Imports

In [1]:
import os
import ast
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cv2
from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms, models
from torchvision.models import ResNet50_Weights, EfficientNet_B0_Weights

from sklearn.model_selection import train_test_split
from sklearn.metrics import (confusion_matrix, accuracy_score,
                             precision_score, recall_score, ConfusionMatrixDisplay)
from scipy import stats
from statsmodels.stats.contingency_tables import mcnemar

# ── Verify GPU ────────────────────────────────────────────────────────────────
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)
if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM:", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), "GB")


Using device: cuda
GPU: NVIDIA RTX A3000 12GB Laptop GPU
VRAM: 12.9 GB


## 2. Configuration — Update Paths Here

In [2]:
# ── UPDATE THESE PATHS TO WHERE YOU SAVED THE ODIR DATASET ──────────────────
CSV_PATH  = r"C:\Users\agilbert3\Downloads\archive (2)\full_df.csv"
TRAIN_DIR = r"C:\Users\agilbert3\Downloads\archive (2)\ODIR-5K\ODIR-5KTraining Images"
TEST_DIR  = r"C:\Users\agilbert3\Downloads\archive (2)\ODIR-5K\ODIR-5KTesting Images"

IMG_SIZE   = 224
BATCH_SIZE = 64     # Safe for 12GB VRAM
EPOCHS     = 100
NUM_CLASSES = 8

CLASS_NAMES = ["Normal(N)", "Diabetes(D)", "Glaucoma(G)", "Cataract(C)",
               "AMD(A)", "Hypertension(H)", "Myopia(M)", "Other(O)"]


## 3. Dataset & DataLoader

In [3]:
class ODIRDataset(Dataset):
    def __init__(self, df, transform=None):
        self.df        = df.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row      = self.df.iloc[idx]
        filename = row["filename"]

        train_path = os.path.join(TRAIN_DIR, filename)
        test_path  = os.path.join(TEST_DIR,  filename)
        path = train_path if os.path.exists(train_path) else test_path

        img = Image.open(path).convert("RGB")
        if self.transform:
            img = self.transform(img)

        label = torch.tensor(row["target"], dtype=torch.float32)
        return img, label

# ── Transforms ────────────────────────────────────────────────────────────────
train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],   # ImageNet mean
                         [0.229, 0.224, 0.225])    # ImageNet std
])

val_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

# ── Load CSV and split ────────────────────────────────────────────────────────
df = pd.read_csv(CSV_PATH)
df["target"] = df["target"].apply(ast.literal_eval)

train_df, val_df = train_test_split(df, test_size=0.2, random_state=42)

train_dataset = ODIRDataset(train_df, transform=train_transform)
val_dataset   = ODIRDataset(val_df,   transform=val_transform)

# ── Weighted sampler to handle class imbalance ───────────────────────────────
label_indices = np.argmax(np.array(train_df["target"].tolist()), axis=1)
class_counts  = np.bincount(label_indices)
class_weights = 1.0 / class_counts
sample_weights = class_weights[label_indices]
sampler = WeightedRandomSampler(sample_weights, len(sample_weights))

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, sampler=sampler,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

print(f"Training samples  : {len(train_dataset)}")
print(f"Validation samples: {len(val_dataset)}")


Training samples  : 5113
Validation samples: 1279


## 4. Model 1 — ResNet50 (Pretrained, Fine-tuned)

**Architecture:** ResNet50 pretrained on ImageNet. We freeze the early layers
(which already detect edges, textures, shapes) and only fine-tune the last
residual block + our custom classification head. This is why training is fast —
most of the network is already trained.

**Custom head layers added:**
- GlobalAveragePooling (built into ResNet)
- Dense 256 → ReLU → Dropout 0.4
- Dense 128 → ReLU → Dropout 0.3  
- Dense 8 → Softmax

**Total layers: 177**
- ResNet50 base: 171 layers
- Custom head: 6 layers (Linear, ReLU, Dropout ×2 sets + final Linear)


In [4]:
class ResNet50Model(nn.Module):
    def __init__(self, num_classes=8):
        super().__init__()
        # Load pretrained ResNet50
        base = models.resnet50(weights=ResNet50_Weights.IMAGENET1K_V2)

        # Freeze all layers except layer4 (last residual block) and fc
        for name, param in base.named_parameters():
            if "layer4" not in name:
                param.requires_grad = False

        # Keep everything except the final FC layer
        self.backbone = nn.Sequential(*list(base.children())[:-1])  # output: (B, 2048, 1, 1)

        # Custom classification head
        self.head = nn.Sequential(
            nn.Flatten(),                          # (B, 2048)
            nn.Linear(2048, 256),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, num_classes)            # logits — softmax applied in loss
        )

    def forward(self, x):
        x = self.backbone(x)
        x = self.head(x)
        return x

model1 = ResNet50Model(num_classes=NUM_CLASSES).to(device)

# Count trainable vs total params
total_params     = sum(p.numel() for p in model1.parameters())
trainable_params = sum(p.numel() for p in model1.parameters() if p.requires_grad)
print(f"Model 1 — ResNet50")
print(f"Total parameters    : {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print(f"Frozen parameters   : {total_params - trainable_params:,}")


Model 1 — ResNet50
Total parameters    : 24,066,504
Trainable parameters: 15,523,208
Frozen parameters   : 8,543,296


## 5. Model 2 — EfficientNet-B0 (Pretrained, Fine-tuned)

**Why EfficientNet instead of a custom CNN?**  
EfficientNet-B0 uses compound scaling — it scales depth, width, and resolution
together rather than just stacking more layers. This makes it significantly more
parameter-efficient than ResNet50 while often achieving higher accuracy.
It's a genuinely different architectural approach (MBConv blocks with squeeze-and-excitation
vs ResNet's residual blocks), making this a meaningful comparison.

**Custom head layers added:**
- GlobalAveragePooling (built into EfficientNet)
- Dense 128 → ReLU → Dropout 0.3
- Dense 8 → Softmax

**Total layers: 82**
- EfficientNet-B0 base: 78 layers
- Custom head: 4 layers (Linear, ReLU, Dropout, Linear)


In [5]:
class EfficientNetModel(nn.Module):
    def __init__(self, num_classes=8):
        super().__init__()
        base = models.efficientnet_b0(weights=EfficientNet_B0_Weights.IMAGENET1K_V1)

        # Freeze all layers except the last conv block
        for name, param in base.named_parameters():
            if "features.8" not in name and "features.7" not in name:
                param.requires_grad = False

        self.backbone = base.features   # (B, 1280, 7, 7)
        self.pool     = nn.AdaptiveAvgPool2d(1)

        self.head = nn.Sequential(
            nn.Flatten(),
            nn.Linear(1280, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        x = self.backbone(x)
        x = self.pool(x)
        x = self.head(x)
        return x

model2 = EfficientNetModel(num_classes=NUM_CLASSES).to(device)

total_params2     = sum(p.numel() for p in model2.parameters())
trainable_params2 = sum(p.numel() for p in model2.parameters() if p.requires_grad)
print(f"Model 2 — EfficientNet-B0")
print(f"Total parameters    : {total_params2:,}")
print(f"Trainable parameters: {trainable_params2:,}")
print(f"Frozen parameters   : {total_params2 - trainable_params2:,}")


Model 2 — EfficientNet-B0
Total parameters    : 4,172,548
Trainable parameters: 1,294,392
Frozen parameters   : 2,878,156


## 6. Training Loop

In [6]:
def train_model(model, train_loader, val_loader, model_name, epochs=EPOCHS):
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.AdamW(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=1e-3, weight_decay=1e-4
    )
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

    history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}
    best_val_acc = 0.0
    best_weights = None

    for epoch in range(epochs):
        # ── Train ────────────────────────────────────────────────────────────
        model.train()
        running_loss, correct, total = 0.0, 0, 0

        for imgs, labels in train_loader:
            imgs, labels = imgs.to(device), labels.to(device)
            label_idx = labels.argmax(dim=1)

            optimizer.zero_grad()
            outputs = model(imgs)
            loss    = criterion(outputs, label_idx)
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * imgs.size(0)
            correct      += (outputs.argmax(1) == label_idx).sum().item()
            total        += imgs.size(0)

        train_loss = running_loss / total
        train_acc  = correct / total

        # ── Validate ─────────────────────────────────────────────────────────
        model.eval()
        val_loss, val_correct, val_total = 0.0, 0, 0

        with torch.no_grad():
            for imgs, labels in val_loader:
                imgs, labels = imgs.to(device), labels.to(device)
                label_idx = labels.argmax(dim=1)
                outputs   = model(imgs)
                loss      = criterion(outputs, label_idx)

                val_loss    += loss.item() * imgs.size(0)
                val_correct += (outputs.argmax(1) == label_idx).sum().item()
                val_total   += imgs.size(0)

        val_loss = val_loss / val_total
        val_acc  = val_correct / val_total
        scheduler.step()

        history["train_loss"].append(train_loss)
        history["train_acc"].append(train_acc)
        history["val_loss"].append(val_loss)
        history["val_acc"].append(val_acc)

        # Save best weights
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_weights = {k: v.clone() for k, v in model.state_dict().items()}

        if (epoch + 1) % 10 == 0:
            print(f"[{model_name}] Epoch {epoch+1:3d}/{epochs} | "
                  f"Train Loss: {train_loss:.4f} Acc: {train_acc:.4f} | "
                  f"Val Loss: {val_loss:.4f} Acc: {val_acc:.4f}")

    # Restore best weights
    model.load_state_dict(best_weights)
    print(f"\n{model_name} — Best Val Accuracy: {best_val_acc:.4f}")
    return history


## 7. Train Both Models

In [ ]:
print("Training Model 1 — ResNet50...")
history1 = train_model(model1, train_loader, val_loader, "ResNet50")


Training Model 1 — ResNet50...


In [ ]:
print("Training Model 2 — EfficientNet-B0...")
history2 = train_model(model2, train_loader, val_loader, "EfficientNet-B0")


## 8. Training Curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, history, label in zip(axes,
                               [history1, history2],
                               ["Model 1: ResNet50", "Model 2: EfficientNet-B0"]):
    ax.plot(history["train_acc"], label="Train Accuracy")
    ax.plot(history["val_acc"],   label="Val Accuracy")
    ax.set_title(f"Accuracy — {label}")
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Accuracy")
    ax.legend()

plt.tight_layout()
plt.show()


## 9. Evaluate Both Models (Confusion Matrix, Accuracy, Precision, Recall)

In [ ]:
def evaluate_model(model, loader, name):
    model.eval()
    all_preds, all_labels = [], []

    with torch.no_grad():
        for imgs, labels in loader:
            imgs   = imgs.to(device)
            outputs = model(imgs)
            preds   = outputs.argmax(dim=1).cpu().numpy()
            true    = labels.argmax(dim=1).numpy()
            all_preds.extend(preds)
            all_labels.extend(true)

    all_preds  = np.array(all_preds)
    all_labels = np.array(all_labels)

    acc  = accuracy_score(all_labels, all_preds)
    prec = precision_score(all_labels, all_preds, average="weighted", zero_division=0)
    rec  = recall_score(all_labels, all_preds, average="weighted", zero_division=0)

    print(f"\n{'='*50}")
    print(f"{name}")
    print(f"{'='*50}")
    print(f"Accuracy : {acc:.4f}")
    print(f"Precision: {prec:.4f}")
    print(f"Recall   : {rec:.4f}")

    cm   = confusion_matrix(all_labels, all_preds)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=CLASS_NAMES)
    fig, ax = plt.subplots(figsize=(9, 9))
    disp.plot(ax=ax, colorbar=False, cmap="Blues")
    ax.set_title(f"Confusion Matrix — {name}")
    plt.tight_layout()
    plt.show()

    return {"accuracy": acc, "precision": prec, "recall": rec,
            "preds": all_preds, "labels": all_labels}

results1 = evaluate_model(model1, val_loader, "Model 1: ResNet50")
results2 = evaluate_model(model2, val_loader, "Model 2: EfficientNet-B0")


## 10. Statistical Comparison

McNemar's test + 95% confidence interval to determine if one model is
statistically better than the other.


In [ ]:
pred1  = results1["preds"]
pred2  = results2["preds"]
true   = results1["labels"]

correct1 = (pred1 == true)
correct2 = (pred2 == true)

b     = np.sum(~correct1 &  correct2)
c     = np.sum( correct1 & ~correct2)
table = [[np.sum(~correct1 & ~correct2), b],
         [c, np.sum(correct1 & correct2)]]

result = mcnemar(table, exact=False, correction=True)
print("McNemar's test statistic:", result.statistic)
print("p-value                 :", result.pvalue)
print("=> Statistically significant difference" if result.pvalue < 0.05
      else "=> No statistically significant difference (p >= 0.05)")

n        = len(true)
acc_diff = results1["accuracy"] - results2["accuracy"]
se_diff  = np.sqrt((results1["accuracy"] * (1 - results1["accuracy"]) +
                    results2["accuracy"] * (1 - results2["accuracy"])) / n)
ci_low, ci_high = acc_diff - 1.96*se_diff, acc_diff + 1.96*se_diff

print(f"\nAccuracy difference (M1 - M2): {acc_diff:+.4f}")
print(f"95% Confidence Interval: [{ci_low:.4f}, {ci_high:.4f}]")


## 11. Real-Time Diagnosis Demo

In [ ]:
def diagnose_image(image_path, model, results, model_name="Model"):
    """Classify a single image and show Type 1 / Type 2 error rates."""
    img = Image.open(image_path).convert("RGB")
    tensor = val_transform(img).unsqueeze(0).to(device)

    model.eval()
    with torch.no_grad():
        probs      = torch.softmax(model(tensor), dim=1).cpu().numpy()[0]
    pred_class = np.argmax(probs)
    confidence = probs[pred_class]

    print(f"[{model_name}] Predicted: {CLASS_NAMES[pred_class]} ({confidence:.2%})")

    # Type 1 / Type 2 error rates for the predicted class
    pred_arr = results["preds"]
    true_arr = results["labels"]

    tp = np.sum((pred_arr == pred_class) & (true_arr == pred_class))
    fp = np.sum((pred_arr == pred_class) & (true_arr != pred_class))
    fn = np.sum((pred_arr != pred_class) & (true_arr == pred_class))

    type1 = fp / (tp + fp) if (tp + fp) > 0 else 0.0
    type2 = fn / (tp + fn) if (tp + fn) > 0 else 0.0
    print(f"Type 1 error rate (false positive): {type1:.2%}")
    print(f"Type 2 error rate (false negative): {type2:.2%}")

    plt.imshow(img)
    plt.title(f"{model_name} — {CLASS_NAMES[pred_class]} ({confidence:.1%})")
    plt.axis("off")
    plt.show()

# ── Demo on a random validation image ────────────────────────────────────────
sample_idx  = np.random.randint(0, len(val_df))
sample_row  = val_df.iloc[sample_idx]
filename    = sample_row["filename"]
path        = os.path.join(TRAIN_DIR, filename) if os.path.exists(os.path.join(TRAIN_DIR, filename)) else os.path.join(TEST_DIR, filename)
true_label  = CLASS_NAMES[np.argmax(sample_row["target"])]

print(f"True label: {true_label}\n")
diagnose_image(path, model1, results1, "ResNet50")
diagnose_image(path, model2, results2, "EfficientNet-B0")


## 12. GradCAM Heatmap — What is the model looking at?

In [ ]:
class GradCAM:
    def __init__(self, model, target_layer):
        self.model       = model
        self.gradients   = None
        self.activations = None

        target_layer.register_forward_hook(
            lambda m, i, o: setattr(self, "activations", o.detach()))
        target_layer.register_backward_hook(
            lambda m, gi, go: setattr(self, "gradients", go[0].detach()))

    def generate(self, img_tensor, class_idx=None):
        self.model.eval()
        img_tensor = img_tensor.unsqueeze(0).to(device)
        output     = self.model(img_tensor)

        if class_idx is None:
            class_idx = output.argmax(dim=1).item()

        self.model.zero_grad()
        output[0, class_idx].backward()

        weights  = self.gradients.mean(dim=(2, 3), keepdim=True)
        heatmap  = (weights * self.activations).sum(dim=1).squeeze()
        heatmap  = torch.clamp(heatmap, min=0)
        heatmap  = heatmap / (heatmap.max() + 1e-8)
        return heatmap.cpu().numpy(), class_idx

def show_gradcam(img_pil, heatmap, title="GradCAM"):
    img_np      = np.array(img_pil.resize((IMG_SIZE, IMG_SIZE))) / 255.0
    heatmap_res = cv2.resize(heatmap, (IMG_SIZE, IMG_SIZE))
    heatmap_col = cv2.applyColorMap(np.uint8(255 * heatmap_res), cv2.COLORMAP_JET)
    heatmap_col = cv2.cvtColor(heatmap_col, cv2.COLOR_BGR2RGB) / 255.0
    overlay     = np.clip(0.6 * img_np + 0.4 * heatmap_col, 0, 1)

    fig, axes = plt.subplots(1, 3, figsize=(14, 5))
    axes[0].imshow(img_np);                      axes[0].set_title("Original");  axes[0].axis("off")
    axes[1].imshow(heatmap_res, cmap="jet");     axes[1].set_title("Heatmap");   axes[1].axis("off")
    axes[2].imshow(overlay);                     axes[2].set_title("Overlay");   axes[2].axis("off")
    plt.suptitle(title)
    plt.tight_layout()
    plt.show()

# ── GradCAM for Model 1 (ResNet50) — target last conv layer ──────────────────
target_layer1 = model1.backbone[-2][-1].conv3   # last conv in layer4
cam1          = GradCAM(model1, target_layer1)

img_pil    = Image.open(path).convert("RGB")
img_tensor = val_transform(img_pil)
heatmap1, pred1_idx = cam1.generate(img_tensor)
show_gradcam(img_pil, heatmap1,
             title=f"GradCAM — ResNet50 | Pred: {CLASS_NAMES[pred1_idx]} | True: {true_label}")

# ── GradCAM for Model 2 (EfficientNet-B0) — target last conv block ───────────
target_layer2 = model2.backbone[-1][0]          # last MBConv block
cam2          = GradCAM(model2, target_layer2)
heatmap2, pred2_idx = cam2.generate(img_tensor)
show_gradcam(img_pil, heatmap2,
             title=f"GradCAM — EfficientNet | Pred: {CLASS_NAMES[pred2_idx]} | True: {true_label}")
